# E1: project the original supplied 3PTB PDBQT representation

Owner: BindingMT #78. This bounded source/editable receiving uses all five saved
poses and the original captured ligand PDBQT. It checks the listed supplied fields while chemical authority remains partial;
it does not recover the missing complete prepared native source or establish
chemical equivalence, energy comparability or ranking.

Supply `BINDINGMT_E1_RESULT` and a new `BINDINGMT_E1_OUTPUT` directory. Verify the
code/input identities and resource prescription before Run All. No preparation,
H generation, charge/type calculation, Vina, interactions or visualization runs.


In [ ]:
import base64
from copy import deepcopy
from functools import wraps
import gzip
import hashlib
import importlib.metadata as metadata
import json
import os
from pathlib import Path
from types import SimpleNamespace
import sys

import numpy as np
import bindingmt as dmt
import molsysmt as msm
import pyunitwizard as puw
from smonitor.integrations import CatalogException

result_path = Path(os.environ['BINDINGMT_E1_RESULT'])
output = Path(os.environ['BINDINGMT_E1_OUTPUT'])
output.mkdir(parents=True, exist_ok=False)
compressed = result_path.read_bytes()
assert hashlib.sha256(compressed).hexdigest() == '4b1f51cdeb135fd6785a181f6bb09a0d0f631ec3d9de7c169c88703748924249'
decoded = gzip.decompress(compressed)
assert hashlib.sha256(decoded).hexdigest() == 'ed0eeb3ade25e54f8a5829729578fcbec9f0d73d6c0ceb03d146b0c3492d7fac'
original = json.loads(decoded)
original_snapshot = deepcopy(original)
capture = original['provenance']['backend_artifacts']['partner']
input_bytes = base64.b64decode(capture['content_base64'], validate=True)
assert hashlib.sha256(input_bytes).hexdigest() == capture['sha256'] == 'a36bd740cc47045c2f2f6756bd8c11b41cbcc960a76ce42c2f2114b3c4b7be46'
input_text = 'pdbqt_text:' + input_bytes.decode('utf-8')
result = dmt.DockingResult.from_dict(original)
assert len(result.poses) == len(original['poses']) == 5
loaded_snapshot = result.to_dict()
(output / 'original_partner.pdbqt').write_bytes(input_bytes)


In [ ]:
# Count direct consumer calls to public entry points, including refusals.
# Provider-internal calls/aliases are outside this counter. No package globals
# are patched; these wrappers only enforce this experiment's finite budget.
budgets = {'convert': 60, 'get': 400, 'copy': 40, 'set': 20,
           'permute_atoms': 4, 'get_atom_correspondences': 2}
counts = dict.fromkeys(budgets, 0)
api = SimpleNamespace()

def counted(name, function):
    @wraps(function)
    def call(*args, **kwargs):
        if counts[name] >= budgets[name]:
            raise RuntimeError('Registered call budget exceeded: ' + name)
        counts[name] += 1
        return function(*args, **kwargs)
    return call

for name in ['convert', 'get', 'copy', 'set', 'permute_atoms']:
    setattr(api, name, counted(name, getattr(msm, name)))
api.get_atom_correspondences = counted(
    'get_atom_correspondences', msm.chemical_states.get_atom_correspondences)

# PDBQT fields are read only by the provider. Its partial charges are explicit
# numeric PDBQT values in elementary charge, independently of session defaults.
names = list(api.get(input_text, element='atom', name=True))
elements = list(api.get(input_text, element='atom', atom_type=True))
ids = list(api.get(input_text, element='atom', atom_id=True))
charges_e = np.asarray(api.get(input_text, element='atom', partial_charge=True), dtype=float)
ff_types = list(api.get(input_text, element='atom', atom_ff_type=True))
tree = msm.form.string_pdbqt_text.get_torsion_tree(input_text)
assert len(names) == len(elements) == len(ids) == len(charges_e) == len(ff_types) == 13
assert tree is not None and tree['torsdof'] == 0
for record in original['poses']:
    assert record['metadata']['pose_atom_order'] == 'verified_pdbqt_order'
    assert record['metadata']['prepared_atom_indices'] == list(range(13))
    assert record['metadata']['selected_atom_indices'] == list(range(13))
    assert [item['atom_name'] for item in record['metadata']['source_atom_keys']] == names
    assert [item['element'] for item in record['metadata']['source_atom_keys']] == elements
# Source-native IDs and PDBQT serials are different recorded axes. They are not
# rewritten to force agreement, and these checks do not certify chemical identity.


In [ ]:
def expected_refusal(name, operation, exception_type, marker):
    try:
        operation()
    except exception_type as error:
        assert marker.lower() in str(error).lower(), str(error)
        return {'control': name, 'disposition': 'refused',
                'exception_type': type(error).__name__,
                'code': getattr(error, 'code', None)}
    raise AssertionError('Expected refusal did not occur: ' + name)

profiles = [
    ('normal', ['nm', 'ps', 'kelvin', 'mole', 'dalton', 'e', 'kJ/mol',
                'kJ/(mol*nm)', 'kJ/(mol*nm**2)', 'radians']),
    ('pm_fs', ['pm', 'fs', 'kelvin', 'mole', 'dalton', 'coulomb', 'kJ/mol',
               'kJ/(mol*pm)', 'kJ/(mol*pm**2)', 'radians']),
]
caller_policy = puw.configure.report()
observations = []
for profile, units in profiles:
    directory = output / profile
    directory.mkdir()
    with puw.context(standard_units=units):
        profile_policy = puw.configure.report()
        base = api.convert(input_text, to_form='molsysmt.MolSys', discard_torsion_tree=True)
        base_xyz = np.asarray(puw.get_value(
            api.get(base, element='atom', coordinates=True), to_unit='angstrom')).copy()
        # Observe only the listed source fields for this nonmutation control;
        # no complete native-state snapshot or persistence claim is made.
        base_fields = {
            'names': list(api.get(base, element='atom', name=True)),
            'ids': list(api.get(base, element='atom', atom_id=True)),
            'types': list(api.get(base, element='atom', atom_ff_type=True)),
        }
        entries = []
        for position, (pose, record) in enumerate(zip(result.poses, original['poses']), 1):
            xyz = np.asarray(puw.get_value(pose.coordinates, to_unit='angstrom'))
            assert xyz.shape == (13, 3) and np.isfinite(xyz).all()
            variant = api.copy(base)
            api.set(variant, element='atom', coordinates=puw.quantity(xyz[None], 'angstrom'))
            exported, mapping = api.convert(
                variant, to_form='string:pdbqt_text', typing_scheme='autodock4',
                torsion_tree=tree, return_atom_map=True)
            payload = exported.removeprefix('pdbqt_text:').encode('utf-8')
            assert mapping['payload_sha256'] == hashlib.sha256(payload).hexdigest()
            order = np.asarray(mapping['output_to_source_atom_indices'], dtype=int)
            np.testing.assert_array_equal(order, np.arange(13))
            np.testing.assert_allclose(puw.get_value(
                api.get(exported, element='atom', coordinates=True), to_unit='angstrom')[0],
                xyz[order], rtol=0, atol=0.000500001)
            np.testing.assert_array_equal(
                api.get(exported, element='atom', atom_ff_type=True),
                np.asarray(ff_types)[order])
            np.testing.assert_allclose(
                api.get(exported, element='atom', partial_charge=True),
                charges_e[order], rtol=0, atol=1e-12)
            np.testing.assert_array_equal(api.get(exported, element='atom', atom_id=True), np.asarray(ids)[order])
            np.testing.assert_array_equal(api.get(exported, element='atom', name=True), np.asarray(names)[order])
            np.testing.assert_array_equal(api.get(exported, element='atom', atom_type=True), np.asarray(elements)[order])
            restored_tree = msm.form.string_pdbqt_text.get_torsion_tree(exported)
            for field in tree:
                np.testing.assert_equal(restored_tree[field], tree[field])
            (directory / f'pose_{position}.pdbqt').write_bytes(payload)
            entries.append({
                'original_position': position, 'pose_id': pose.pose_id,
                'partner_state_id': pose.partner_state_id,
                'receptor_state_id': pose.receptor_state_id,
                'original_scores': deepcopy(record['scores']),
                'output_sha256': mapping['payload_sha256'],
                'output_to_source_atom_indices': order.tolist(),
                'source_atom_ids': np.asarray(mapping['source_atom_ids']).tolist(),
                'output_atom_ids': np.asarray(mapping['output_atom_ids']).tolist(),
                'representation_preservation': 'verified_for_listed_fields',
                'verified_fields': ['coordinates', 'atom_ff_type', 'partial_charge',
                                    'atom_ids', 'atom_names', 'elements', 'torsion_tree', 'output_map'],
                'chemical_identity': 'unassessed_partial_reference',
                'score_comparison': 'not_admitted',
            })
        refusals = []
        refusals.append(expected_refusal(
            'partial_reference_cannot_certify_full_chemical_identity',
            lambda: api.get_atom_correspondences(
                base, structure_indices=[0], chemical_state='reference',
                candidate_atom_correspondences=np.column_stack((np.arange(13), np.arange(13))),
                max_matches=1), msm.StructuralInconsistencyError, 'complete'))
        reversed_source = api.permute_atoms(base, atom_indices=list(range(12, -1, -1)))
        refusals.append(expected_refusal(
            'stale_tree_axis_after_public_permutation',
            lambda: api.convert(reversed_source, to_form='string:pdbqt_text',
                typing_scheme='autodock4', torsion_tree=tree, return_atom_map=True),
            CatalogException, 'atom_ids'))
        nonfinite = api.copy(base)
        invalid_xyz = base_xyz.copy()
        invalid_xyz[0, 0, 0] = np.nan
        api.set(nonfinite, element='atom', coordinates=puw.quantity(invalid_xyz, 'angstrom'))
        refusals.append(expected_refusal(
            'nonfinite_geometry_not_exported',
            lambda: api.convert(nonfinite, to_form='string:pdbqt_text',
                typing_scheme='autodock4', torsion_tree=tree, return_atom_map=True),
            CatalogException, 'finite'))
        np.testing.assert_array_equal(puw.get_value(
            api.get(base, element='atom', coordinates=True), to_unit='angstrom'), base_xyz)
        assert list(api.get(base, element='atom', name=True)) == base_fields['names']
        assert list(api.get(base, element='atom', atom_id=True)) == base_fields['ids']
        assert list(api.get(base, element='atom', atom_ff_type=True)) == base_fields['types']
        np.testing.assert_allclose(
            api.get(base, element='atom', partial_charge=True), charges_e, rtol=0, atol=1e-12)
        assert puw.configure.report() == profile_policy
        profile_record = {'profile': profile, 'poses': entries, 'controls': refusals,
                          'source_nonmutation_verified_fields': ['coordinates', 'atom_names', 'atom_ids',
                              'atom_ff_type', 'partial_charge'], 'unit_policy_preserved': True}
        (directory / 'observations.json').write_text(json.dumps(profile_record, indent=2) + '\n')
        observations.append(profile_record)
    assert puw.configure.report() == caller_policy
    assert original == original_snapshot
    assert result.to_dict() == loaded_snapshot
    assert result_path.read_bytes() == compressed


In [ ]:
assert [pose['output_sha256'] for pose in observations[0]['poses']] == [
    pose['output_sha256'] for pose in observations[1]['poses']]
assert counts['get_atom_correspondences'] == 2
assert not any(name == 'vina' or name.startswith('vina.') for name in sys.modules)
receipt = {
    'schema': 'bindingmt.e1_partial_projection_receiving@1',
    'owner': 'uibcdf/bindingmt#78',
    'scope': 'Supplied partial PDBQT projection only; no full chemical or comparison admission',
    'versions': {name: metadata.version(name) for name in ['bindingmt', 'molsysmt', 'argdigest', 'pyunitwizard', 'numpy', 'rdkit']},
    'original_result_compressed_sha256': hashlib.sha256(compressed).hexdigest(),
    'original_result_decoded_sha256': hashlib.sha256(decoded).hexdigest(),
    'original_input_sha256': capture['sha256'],
    'original_producer': {key: original['provenance'][key] for key in ['bindingmt_version', 'molsysmt_version', 'backend', 'backend_version']},
    'original_pose_count': 5, 'distinct_poses': 5,
    'attempted_profiles': len(observations), 'completed_profiles': len(observations),
    'projected_pose_records': sum(len(item['poses']) for item in observations),
    'expected_refusals': sum(len(item['controls']) for item in observations),
    'profiles': observations, 'direct_consumer_public_entry_point_calls': counts,
    'direct_consumer_public_entry_point_budgets': budgets,
    'cross_policy_payloads_equal': True,
    'original_records_and_captures_preserved': True,
    'full_native_source_restored': False,
    'chemical_equivalence_admitted': False,
    'e1_accepted': False,
    'excluded_operations': {'direct_calls_in_reviewed_notebook': 0,
        'groups': ['searches', 'scores', 'preparations', 'H_generation',
                   'charge_type_calculation', 'detectors', 'viewer'],
        'basis': 'Reviewed notebook route; these excluded APIs are not counter-instrumented'},
    'claim_limit': 'These are representation-preservation controls over one original case; repeated unit profiles are not independent poses or complexes.'
}
(output / 'receipt.json').write_text(json.dumps(receipt, indent=2) + '\n')
print(json.dumps({'profiles': 2, 'distinct_poses': 5, 'projections': 10,
                  'expected_refusals': 6, 'chemical_identity': 'unassessed',
                  'energy_calls': 0}))


The original capture retains its original preparation REMARKs and producer.
The new provider writer canonicalizes presentation; original REMARKs are not
relabelled as fresh preparation or scoring evidence. The per-pose input hashes,
maps, original scores and explicit unknown chemical authority remain separate.

The retained native 13-atom complete chemical source is unavailable in the
inspected custody set. The later 18-atom E4 ligand is a different representation
and is not used as a replacement here. Full-state projection, flexible and
nonidentity export receiving, positive chemical identity, comparison admission
and representative native/installed rescoring remain separate E1 work.
